In [13]:
import sys, tensorflow as tf, cv2, sklearn, mlflow, ultralytics
print("Python      :", sys.version.split()[0])
print("TensorFlow  :", tf.__version__)
print("OpenCV      :", cv2.__version__)
print("scikit-learn:", sklearn.__version__)
print("MLflow      :", mlflow.__version__)
print("Ultralytics :", ultralytics.__version__)
print("GPUs        :", tf.config.list_physical_devices("GPU"))

Python      : 3.12.10
TensorFlow  : 2.21.0
OpenCV      : 5.0.0
scikit-learn: 1.9.1
MLflow      : 3.17.2
Ultralytics : 8.4.183
GPUs        : []


In [14]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError
import pandas as pd
import shutil
from tqdm import tqdm

# If this path is wrong, hardcode it:
# PROJECT_ROOT = Path(r"C:\Users\ycode\Desktop\brain-tumor-project")
PROJECT_ROOT = Path.cwd().parent        # notebook is in notebooks/

RAW_DIR   = PROJECT_ROOT / "data" / "classification" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "classification" / "clean"

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

print("cwd          :", Path.cwd())
print("RAW_DIR      :", RAW_DIR)
print("RAW_DIR exists:", RAW_DIR.exists())

all_files = [p for p in RAW_DIR.rglob("*") if p.is_file()] if RAW_DIR.exists() else []
print("Total files found:", len(all_files))

assert all_files, (
    f"No files found in {RAW_DIR}.\n"
    "Copy your dataset there (one folder per class) and re-run this cell."
)

print("\nFolders containing files:")
print(pd.Series([p.parent.name for p in all_files]).value_counts())

cwd          : c:\Users\ycode\Desktop\brain-tumor-project\notebooks
RAW_DIR      : c:\Users\ycode\Desktop\brain-tumor-project\data\classification\raw
RAW_DIR exists: True
Total files found: 7023

Folders containing files:
notumor       2000
pituitary     1757
meningioma    1645
glioma        1621
Name: count, dtype: int64


In [15]:
records = []

for file in tqdm(all_files, desc="Checking files"):
    status, reason, size = "valid", "", None

    # 1. Extension check
    if file.suffix.lower() not in VALID_EXT:
        status, reason = "invalid", f"bad extension ({file.suffix})"
    else:
        # 2. Integrity check
        try:
            with Image.open(file) as img:
                img.verify()                  # detects truncated/corrupted files
            with Image.open(file) as img:     # reopen: verify() invalidates the object
                img.load()
                size = img.size
        except (UnidentifiedImageError, OSError, ValueError, SyntaxError) as e:
            status, reason = "invalid", f"corrupted ({type(e).__name__})"

    records.append({
        "path": str(file),
        "class": file.parent.name,            # class = name of the parent folder
        "status": status,
        "reason": reason,
        "size": size,
    })

df = pd.DataFrame(records)
print(df["status"].value_counts())
print("\nClasses:", sorted(df["class"].unique()))
df[df["status"] == "invalid"].head(20)

Checking files:   0%|          | 0/7023 [00:00<?, ?it/s]

Checking files: 100%|██████████| 7023/7023 [00:17<00:00, 404.48it/s]

status
valid    7023
Name: count, dtype: int64

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


,path,class,status,reason,size


In [27]:
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

IMG_SIZE = (299, 299)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def load(name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / name,
        image_size=IMG_SIZE,
        batch_size=BATCH,
        color_mode="rgb",
        label_mode="categorical",   # one-hot labels
        shuffle=shuffle,            # True for train only
        seed=42,
    )

train_ds = load("train", shuffle=True)
val_ds   = load("val",   shuffle=False)
test_ds  = load("test",  shuffle=False)

class_names = train_ds.class_names
print("Classes:", class_names)

# Normalize for InceptionV3 (scales pixels to [-1, 1])
def prep(x, y):
    return preprocess_input(x), y

train_ds = train_ds.map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(prep,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(prep,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

Found 4872 files belonging to 4 classes.
Found 970 files belonging to 4 classes.
Found 971 files belonging to 4 classes.
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [28]:
from tensorflow.keras import layers, models, optimizers, callbacks

# 1. Load InceptionV3 without its classification head and freeze it
base_model = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=(299, 299, 3),
)
base_model.trainable = False

# 2. New head for our 4 classes
inputs  = layers.Input(shape=(299, 299, 3))
x       = base_model(inputs, training=False)   # keeps BatchNorm in inference mode
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = models.Model(inputs, outputs)

# 3. Hyperparameters
LR = 1e-3
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# 4. Callbacks
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    callbacks.ModelCheckpoint(MODELS_DIR / "inceptionv3_fe.keras", monitor="val_loss", save_best_only=True),
]

model.summary()

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 299, 299, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 8, 8, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,810,980 (83.20 MB)

 Trainable params: 8,196 (32.02 KB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=cbs
)

#Learning curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history["accuracy"], label="train")
ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].plot(history.history["loss"], label="train")
ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss")
ax[1].set_xlabel("Epoch")
ax[1].legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "figures" / "fe_learning_curves.png", dpi=150)
plt.show()

In [29]:
shutil.make_archive(str(PROJECT_ROOT / "split_data"), "zip", SPLIT_DIR)
print("Created:", PROJECT_ROOT / "split_data.zip")

Created: c:\Users\ycode\Desktop\brain-tumor-project\split_data.zip


In [30]:
import tensorflow as tf

MODEL_PATH = PROJECT_ROOT / "models" / "inceptionv3_ft.keras"
final_model = tf.keras.models.load_model(MODEL_PATH)

final_model.summary()

# Save the class names, the API will need them later
import json
with open(PROJECT_ROOT / "models" / "class_names.json", "w") as f:
    json.dump(["glioma", "meningioma", "notumor", "pituitary"], f)

print("Model loaded OK:", MODEL_PATH)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 299, 299, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 8, 8, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,163,374 (137.95 MB)

 Trainable params: 7,176,196 (27.38 MB)

 Non-trainable params: 14,634,784 (55.83 MB)

 Optimizer params: 14,352,394 (54.75 MB)

Model loaded OK: c:\Users\ycode\Desktop\brain-tumor-project\models\inceptionv3_ft.keras
